# Phase 1

# Cell 1 - Imports

In [ ]:
import sys, io


import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import Counter

warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("[OK] Libraries imported successfully.")

# CELL 2  -  Configuration (EDIT THESE PATHS)

In [ ]:
# ── KAGGLE PATH CONFIG (replaces Cell 2 in phase_1.ipynb) ──────────────────
import os
from pathlib import Path

BASE_KAGGLE   = Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")
METADATA_PATH = BASE_KAGGLE / "HAM10000_metadata.csv"
IMAGE_DIR_PART1 = BASE_KAGGLE / "HAM10000_images_part_1"
IMAGE_DIR_PART2 = BASE_KAGGLE / "HAM10000_images_part_2"

OUTPUT_DIR = Path("/kaggle/working/phase1_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR = OUTPUT_DIR / "splits"
SPLITS_DIR.mkdir(exist_ok=True)

IMG_SIZE = (224, 224)
print(f"[OK] Kaggle paths set. Dataset at: {BASE_KAGGLE}")
print(f"[OK] Outputs -> {OUTPUT_DIR}")


# CELL 3  -  STEP 1.1: Load & Validate Dataset

In [ ]:
# ── 3A. Load the raw metadata ──────────────────────────────────────────────────
df = pd.read_csv(METADATA_PATH)
print(f"Raw dataset loaded: {df.shape[0]} records, {df.shape[1]} columns")
print(df.head(3))

# ── 3B. Build image path map (combining both parts) ────────────────────────────
image_path_map = {}
for folder in [IMAGE_DIR_PART1, IMAGE_DIR_PART2]:
    if folder.exists():
        for img_file in folder.glob("*.jpg"):
            image_path_map[img_file.stem] = str(img_file)

df["image_path"] = df["image_id"].map(image_path_map)
df["image_exists"] = df["image_path"].notnull()

missing_images = (~df["image_exists"]).sum()
print(f"\n[DIR] Total images found across both folders : {len(image_path_map)}")
print(f"[OK] Metadata rows with found images         : {df['image_exists'].sum()}")
print(f"[WARN]  Metadata rows with MISSING images       : {missing_images}")

# Keep only rows where image actually exists
df = df[df["image_exists"]].copy()
print(f"\n[CHART] Working dataset (images confirmed): {len(df)} records")

# CELL 4  -  STEP 1.1: Data Cleaning (continue from Sem 1)

In [ ]:
print("=" * 55)
print("  DATA QUALITY BEFORE CLEANING")
print("=" * 55)
print(df.isnull().sum())

# ── 4A. Handle missing values (same approach as Sem 1) ─────────────────────────
#   age  -> median imputation
#   sex  -> mode imputation
#   localization -> mode imputation

AGE_MEDIAN = df["age"].median()
SEX_MODE   = df["sex"].mode()[0]
LOC_MODE   = df["localization"].mode()[0]

df["age"]          = df["age"].fillna(AGE_MEDIAN)
df["sex"]          = df["sex"].fillna(SEX_MODE)
df["localization"] = df["localization"].fillna(LOC_MODE)

# ── 4B. Drop 'unknown' sex rows (only 3 rows in HAM10000) ──────────────────────
df = df[df["sex"] != "unknown"].copy()

print("\n[OK] After cleaning:")
print(df.isnull().sum())
print(f"\nFinal clean dataset: {len(df)} records")

# CELL 5  -  STEP 1.1: Create Engineered Features

In [ ]:
# ── 5A. Map dx codes to full names ─────────────────────────────────────────────
dx_mapping = {
    "akiec": "Actinic Keratoses",
    "bcc"  : "Basal Cell Carcinoma",
    "bkl"  : "Benign Keratosis",
    "df"   : "Dermatofibroma",
    "mel"  : "Melanoma",
    "nv"   : "Melanocytic Nevi",
    "vasc" : "Vascular Lesions"
}
df["dx_full"] = df["dx"].map(dx_mapping)

# ── 5B. Create age groups (4 clinical bins) ─────────────────────────────────────
#   Pediatric   : < 18
#   Young Adult : 18 - 39
#   Middle-Aged : 40 - 59
#   Elderly     : 60+

def assign_age_group(age):
    if age < 18:
        return "Pediatric"
    elif age < 40:
        return "YoungAdult"
    elif age < 60:
        return "MiddleAged"
    else:
        return "Elderly"

df["age_group"] = df["age"].apply(assign_age_group)

# ── 5C. Standardise sex labels ──────────────────────────────────────────────────
df["sex"] = df["sex"].str.lower().str.strip()

# ── 5D. Simplify localization into broader anatomical zones ─────────────────────
loc_zone_map = {
    "back"          : "Trunk",
    "lower extremity": "LowerExtremity",
    "trunk"         : "Trunk",
    "upper extremity": "UpperExtremity",
    "abdomen"       : "Trunk",
    "face"          : "Head",
    "chest"         : "Trunk",
    "foot"          : "LowerExtremity",
    "unknown"       : "Unknown",
    "neck"          : "Head",
    "scalp"         : "Head",
    "hand"          : "UpperExtremity",
    "ear"           : "Head",
    "genital"       : "Trunk",
    "acral"         : "LowerExtremity"
}
df["loc_zone"] = df["localization"].map(loc_zone_map).fillna("Other")

# ── 5E. Create the INTERSECTIONAL KEY (core of Framework Step 1.2) ─────────────
#   Key = age_group|sex|loc_zone
#   e.g.  "Elderly|male|Trunk" , "Pediatric|female|Head"
df["intersectional_key"] = (
    df["age_group"] + "|" + df["sex"] + "|" + df["loc_zone"]
)

print("[OK] Feature engineering complete.")
print(f"\nUnique dx classes    : {df['dx'].nunique()}")
print(f"Unique age groups    : {df['age_group'].nunique()} -> {sorted(df['age_group'].unique())}")
print(f"Unique sex values    : {df['sex'].nunique()} -> {df['sex'].unique().tolist()}")
print(f"Unique loc zones     : {df['loc_zone'].nunique()} -> {sorted(df['loc_zone'].unique())}")
print(f"Unique intersect keys: {df['intersectional_key'].nunique()}")

print(df[["image_id","dx","age_group","sex","loc_zone","intersectional_key"]].head(8))

# CELL 6  -  STEP 1.1: Baseline Bias Audit (pre-framework)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("[CHART] BIAS AUDIT  -  HAM10000 Dataset (Pre-Framework)", fontsize=16, fontweight="bold")

# ── Plot 1: Class distribution ──────────────────────────────────────────────────
ax = axes[0, 0]
dx_counts = df["dx"].value_counts()
bars = ax.bar(dx_counts.index, dx_counts.values,
              color=sns.color_palette("tab10", len(dx_counts)))
ax.set_title("Diagnosis Class (dx) Distribution", fontweight="bold")
ax.set_xlabel("Diagnosis Code")
ax.set_ylabel("Count")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(bar.get_height()), ha="center", va="bottom", fontsize=10)

# ── Plot 2: Age group distribution ─────────────────────────────────────────────
ax = axes[0, 1]
age_counts = df["age_group"].value_counts()
ax.pie(age_counts, labels=age_counts.index, autopct="%1.1f%%",
       colors=sns.color_palette("pastel", len(age_counts)), startangle=90,
       wedgeprops={"edgecolor": "white", "linewidth": 2})
ax.set_title("Age Group Distribution", fontweight="bold")

# ── Plot 3: Sex distribution ────────────────────────────────────────────────────
ax = axes[1, 0]
sex_counts = df["sex"].value_counts()
ax.bar(sex_counts.index.str.capitalize(), sex_counts.values,
       color=["#3498db", "#e74c3c"])
ax.set_title("Sex Distribution", fontweight="bold")
ax.set_ylabel("Count")

# ── Plot 4: Intersectional key  -  top 20 most/least represented ─────────────────
ax = axes[1, 1]
key_counts = df["intersectional_key"].value_counts()
top_keys = pd.concat([key_counts.head(10), key_counts.tail(10)])
colors = ["#27ae60"] * 10 + ["#e74c3c"] * 10
ax.barh(range(len(top_keys)), top_keys.values, color=colors)
ax.set_yticks(range(len(top_keys)))
ax.set_yticklabels(top_keys.index, fontsize=8)
ax.set_title("Intersectional Keys  -  Top 10 Over & Under Represented", fontweight="bold")
ax.set_xlabel("Count")
# Legend
over_patch  = mpatches.Patch(color="#27ae60", label="Over-represented (top 10)")
under_patch = mpatches.Patch(color="#e74c3c", label="Under-represented (bottom 10)")
ax.legend(handles=[over_patch, under_patch], loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bias_audit_pre_framework.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"[OK] Bias audit chart saved -> {OUTPUT_DIR / 'bias_audit_pre_framework.png'}")

# CELL 7  -  STEP 1.2: Intersectional Stratified Sampling

In [ ]:
print("=" * 60)
print("  STEP 1.2  -  INTERSECTIONAL STRATIFIED SAMPLING")
print("=" * 60)

# ── 7A. Encode the target (dx) for stratification ──────────────────────────────
le_dx = LabelEncoder()
df["dx_encoded"] = le_dx.fit_transform(df["dx"])

# Store class name mapping
class_names = list(le_dx.classes_)
print(f"Classes (encoded 0-{len(class_names)-1}): {class_names}")

# ── 7B. Build a combined stratification key: dx + intersectional_key ─────────
#   This ensures the split is balanced across BOTH diagnosis AND demographics
df["strat_key"] = df["dx"] + "__" + df["intersectional_key"]

# Some rare strat_key combos may have only 1 sample -> cannot split those
# Identify and handle singleton groups
strat_counts = df["strat_key"].value_counts()
singleton_keys = strat_counts[strat_counts == 1].index.tolist()

if singleton_keys:
    print(f"\n[WARN]  {len(singleton_keys)} singleton strat_key groups found.")
    print("   These will be kept in training set only (cannot be split).")
    df_singletons = df[df["strat_key"].isin(singleton_keys)].copy()
    df_splittable = df[~df["strat_key"].isin(singleton_keys)].copy()
else:
    df_singletons = pd.DataFrame()
    df_splittable = df.copy()

print(f"\nSplittable records: {len(df_splittable)}")
print(f"Singleton records   : {len(df_singletons)}")

# ── 7C. Perform the stratified split: 70% train | 15% val | 15% test ───────────
#   First split off 30% for val+test, then split that 50/50

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_SEED)
train_idx, temp_idx = next(sss1.split(df_splittable, df_splittable["strat_key"]))

df_train = df_splittable.iloc[train_idx].copy()
df_temp  = df_splittable.iloc[temp_idx].copy()

# Handle singleton strat_keys in df_temp for second split
strat_counts_temp = df_temp["strat_key"].value_counts()
singleton_temp    = strat_counts_temp[strat_counts_temp == 1].index.tolist()
df_temp_singletons = df_temp[df_temp["strat_key"].isin(singleton_temp)].copy()
df_temp_splittable = df_temp[~df_temp["strat_key"].isin(singleton_temp)].copy()

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_SEED)
val_idx, test_idx = next(sss2.split(df_temp_splittable, df_temp_splittable["strat_key"]))

df_val  = df_temp_splittable.iloc[val_idx].copy()
df_test = df_temp_splittable.iloc[test_idx].copy()

# Add singletons to training set
df_train = pd.concat([df_train, df_singletons, df_temp_singletons], ignore_index=True)

print(f"\n[CHART] SPLIT SUMMARY:")
print(f"  Training set  : {len(df_train):>5} records ({len(df_train)/len(df)*100:.1f}%)")
print(f"  Validation set: {len(df_val):>5} records ({len(df_val)/len(df)*100:.1f}%)")
print(f"  Test set      : {len(df_test):>5} records ({len(df_test)/len(df)*100:.1f}%)")
print(f"  Total         : {len(df_train)+len(df_val)+len(df_test):>5} records")


# ── 7D. Verify stratification quality ──────────────────────────────────────────
print("\n[CHART] CLASS DISTRIBUTION ACROSS SPLITS (%):")

split_comparison = pd.DataFrame({
    "Full Dataset": df["dx"].value_counts(normalize=True) * 100,
    "Train Split" : df_train["dx"].value_counts(normalize=True) * 100,
    "Val Split"   : df_val["dx"].value_counts(normalize=True) * 100,
    "Test Split"  : df_test["dx"].value_counts(normalize=True) * 100,
}).round(2)
print(split_comparison)

# Visualise split proportions
fig, ax = plt.subplots(figsize=(12, 5))
split_comparison.plot(kind="bar", ax=ax, color=["#95a5a6","#3498db","#2ecc71","#e74c3c"],
                      edgecolor="white", linewidth=0.5)
ax.set_title("Diagnosis Class Distribution Across Train/Val/Test Splits",
             fontweight="bold", fontsize=14)
ax.set_xlabel("Diagnosis Class (dx)")
ax.set_ylabel("Percentage (%)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "stratified_split_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"[OK] Split distribution chart saved.")

# CELL 8  -  STEP 1.3: Adaptive Distribution-Aware Reweighting

In [ ]:
print("=" * 60)
print("  STEP 1.3  -  ADAPTIVE DISTRIBUTION-AWARE REWEIGHTING")
print("=" * 60)

# ── 8A. Compute inverse-frequency weights based on INTERSECTIONAL KEY ───────────
#   Weight formula:  w_i = N_total / (N_unique_keys × count_of_key_i)
#   This gives higher weights to rare demographic subgroups

# Compute on the TRAINING SET only (to avoid data leakage)
key_counts_train = df_train["intersectional_key"].value_counts().to_dict()
N_total  = len(df_train)
N_keys   = len(key_counts_train)

df_train["sample_weight"] = df_train["intersectional_key"].map(
    lambda key: N_total / (N_keys * key_counts_train[key])
)

# ── 8B. Normalize weights so the mean = 1.0 (preserves effective sample size) ──
mean_weight = df_train["sample_weight"].mean()
df_train["sample_weight"] = df_train["sample_weight"] / mean_weight

print(f"\nWeight statistics (training set):")
print(df_train["sample_weight"].describe().round(4))

# ── 8C. Apply same weight logic to val/test (using train key counts to avoid leakage) ─
#   If a key exists in val/test but not train -> assign max_train_weight
max_train_weight = df_train["sample_weight"].max()

def compute_weight(key, key_counts, N_total, N_keys, max_w):
    if key in key_counts:
        return (N_total / (N_keys * key_counts[key])) / mean_weight
    return max_w

df_val["sample_weight"]  = df_val["intersectional_key"].apply(
    lambda k: compute_weight(k, key_counts_train, N_total, N_keys, max_train_weight))
df_test["sample_weight"] = df_test["intersectional_key"].apply(
    lambda k: compute_weight(k, key_counts_train, N_total, N_keys, max_train_weight))

# ── 8D. Visualise weight distribution ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Weight histogram
axes[0].hist(df_train["sample_weight"], bins=50, color="#3498db", edgecolor="white",
             linewidth=0.5)
axes[0].axvline(1.0, color="#e74c3c", linestyle="--", linewidth=2, label="Mean Weight (1.0)")
axes[0].set_title("Sample Weight Distribution (Training Set)", fontweight="bold")
axes[0].set_xlabel("Sample Weight")
axes[0].set_ylabel("Frequency")
axes[0].legend()

# Average weight per dx class
avg_weight_per_dx = df_train.groupby("dx")["sample_weight"].mean().sort_values(ascending=False)
colors = ["#e74c3c" if w > 1.5 else "#f39c12" if w > 1.0 else "#27ae60"
          for w in avg_weight_per_dx.values]
axes[1].bar(avg_weight_per_dx.index, avg_weight_per_dx.values, color=colors, edgecolor="white")
axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1.5, label="Baseline weight = 1.0")
axes[1].set_title("Average Sample Weight per Diagnosis Class (Training Set)",
                  fontweight="bold")
axes[1].set_xlabel("Diagnosis Class (dx)")
axes[1].set_ylabel("Average Sample Weight")
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_weights_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# ── 8E. Print top-weighted (most underrepresented) subgroups ───────────────────
print("\n[HIGH] TOP 15 MOST UNDERREPRESENTED SUBGROUPS (highest weights):")
top_underrep = (
    df_train.groupby("intersectional_key")["sample_weight"]
    .agg(["mean","count"])
    .sort_values("mean", ascending=False)
    .head(15)
)
top_underrep.columns = ["Average Weight", "Sample Count"]
print(top_underrep)

print("\n[LOW] TOP 15 MOST OVER-REPRESENTED SUBGROUPS (lowest weights):")
top_overrep = (
    df_train.groupby("intersectional_key")["sample_weight"]
    .agg(["mean","count"])
    .sort_values("mean", ascending=True)
    .head(15)
)
top_overrep.columns = ["Average Weight", "Sample Count"]
print(top_overrep)

# CELL 9  -  Identify Rare Subgroups for cGAN Augmentation

In [ ]:
print("=" * 60)
print("  IDENTIFYING RARE SUBGROUPS -> cGAN Augmentation Targets")
print("=" * 60)

# ── Define rare subgroup threshold: key with < RARE_THRESHOLD samples ───────────
RARE_THRESHOLD = 30  # subgroups with fewer than 30 training samples

key_counts_df = (
    df_train.groupby(["intersectional_key","dx","age_group","sex","loc_zone"])
    .size()
    .reset_index(name="count")
    .sort_values("count")
)

rare_subgroups = key_counts_df[key_counts_df["count"] < RARE_THRESHOLD].copy()
print(f"\n[TARGET] Found {len(rare_subgroups)} rare subgroups (< {RARE_THRESHOLD} samples):")
print(rare_subgroups)

# Save for cGAN notebook (Phase 1 Step 2)
rare_subgroups.to_csv(OUTPUT_DIR / "rare_subgroups_cgan_targets.csv", index=False)
print(f"\n[OK] Rare subgroups saved -> {OUTPUT_DIR / 'rare_subgroups_cgan_targets.csv'}")

# Summary per dx class
print("\n[CHART] Rare subgroup count per dx class:")
print(rare_subgroups.groupby("dx")["count"].agg(["sum","count"])
        .rename(columns={"sum":"Total samples in rare subgroups",
                         "count":"Number of rare subgroups"}))

# CELL 10  -  Save All Processed Splits

In [ ]:
# ── Save split DataFrames to CSV for use in cGAN & CNN notebooks ────────────────
df_train.to_csv(SPLITS_DIR / "train_split.csv", index=False)
df_val.to_csv(  SPLITS_DIR / "val_split.csv",   index=False)
df_test.to_csv( SPLITS_DIR / "test_split.csv",  index=False)

# ── Save the label encoder mapping ─────────────────────────────────────────────
label_map = {cls: int(idx) for idx, cls in enumerate(le_dx.classes_)}
with open(OUTPUT_DIR / "label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

# ── Save the intersectional key weight map (for CNN training) ──────────────────
weight_map = df_train.set_index("image_id")["sample_weight"].to_dict()
with open(OUTPUT_DIR / "sample_weights.json", "w") as f:
    json.dump(weight_map, f, indent=2)

print("[OK] ALL PHASE 1 STEP 1 OUTPUTS SAVED:")
print(f"  [DIR] {SPLITS_DIR / 'train_split.csv'}  ({len(df_train)} rows)")
print(f"  [DIR] {SPLITS_DIR / 'val_split.csv'}    ({len(df_val)} rows)")
print(f"  [DIR] {SPLITS_DIR / 'test_split.csv'}   ({len(df_test)} rows)")
print(f"  [DIR] {OUTPUT_DIR / 'label_map.json'}")
print(f"  [DIR] {OUTPUT_DIR / 'sample_weights.json'}")
print(f"  [DIR] {OUTPUT_DIR / 'rare_subgroups_cgan_targets.csv'}")

# CELL 11  -  Final Summary Report

In [ ]:
print("\n" + "=" * 60)
print("  PHASE 1 STEPS 1.1-1.3  -  COMPLETE SUMMARY")
print("=" * 60)

print(f"""
DATASET PREPARATION (Step 1.1):
  |-- Total records loaded           : {len(df):,}
  |-- Images verified & linked       : {len(df):,}
  |-- Missing values handled         : Median (age), Mode (sex, localization)
  `-- Age groups created             : Pediatric / YoungAdult / MiddleAged / Elderly

INTERSECTIONAL STRATIFIED SAMPLING (Step 1.2):
  |-- Unique intersectional keys     : {df['intersectional_key'].nunique()}
  |-- Training set                   : {len(df_train):,} records
  |-- Validation set                 : {len(df_val):,} records
  `-- Test set                       : {len(df_test):,} records

ADAPTIVE DISTRIBUTION-AWARE REWEIGHTING (Step 1.3):
  |-- Weight range (training)        : {df_train['sample_weight'].min():.3f} - {df_train['sample_weight'].max():.3f}
  |-- Mean weight (normalised to)    : {df_train['sample_weight'].mean():.3f}
  `-- Rare subgroups for cGAN aug.   : {len(rare_subgroups)} subgroups (< {RARE_THRESHOLD} samples)

NEXT STEP -> Open: phase1_step2_cgan_training.py
""")


# Phase 2

# Cell 0

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 2 — CELL 0: GPU Environment Setup  (ONE-TIME ONLY)
# This cell kills the kernel to force a clean restart.
# After the restart dialog appears → click Ok →
# then run from Phase 1 Cell 1 downward, SKIPPING this cell.
# ═══════════════════════════════════════════════════════════════
import subprocess, sys, os

def get_torch_version():
    try:
        import torch
        return torch.__version__
    except Exception:
        return None

REQUIRED_TORCH = '2.2.2'
current = get_torch_version()

if current and current.startswith(REQUIRED_TORCH):
    # Already correct version — no install or restart needed
    print(f'[OK] torch {current} already installed. No restart needed.')
    print('[OK] You can run the remaining cells directly.')
else:
    print(f'[INFO] Current torch: {current} — installing correct version...')

    # Step 1 — Pin NumPy to 1.x before torch install
    print('[STEP 1] Pinning NumPy < 2.0...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'numpy<2.0', '--quiet'
    ], stderr=subprocess.DEVNULL)
    print('[STEP 1] Done.')

    # Step 2 — Install matching torch for Kaggle P100 GPU (SM 6.0, CUDA 12.1)
    print('[STEP 2] Installing torch 2.2.2+cu121 + torchvision 0.17.2+cu121...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'torch==2.2.2+cu121',
        'torchvision==0.17.2+cu121',
        'scipy',
        '--index-url', 'https://download.pytorch.org/whl/cu121',
        '--quiet'
    ], stderr=subprocess.DEVNULL)
    print('[STEP 2] Install complete.')

    # Step 3 — Kill kernel for clean restart
    # The "Kernel Restarting" dialog that appears is NORMAL — click Ok.
    # After restart: run from Phase 1 Cell 1 downward. SKIP this cell.
    print('[STEP 3] Restarting kernel for clean package load...')
    print('>>> When the dialog appears, click OK, then skip this cell and run from Cell 1. <<<')
    os.kill(os.getpid(), 9)

# Cell 1 - Imports

In [ ]:
import sys, io

import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# Kaggle GPU env ships with torch+CUDA pre-installed — do NOT reinstall from PyPI
# Reinstalling from default PyPI replaces the CUDA-matched build with a wrong CUDA wheel
# causing: AcceleratorError: no kernel image is available for execution on the device
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torchvision.models import inception_v3
from scipy import linalg

warnings.filterwarnings("ignore")

# Verify versions are compatible
print(f"torch      : {torch.__version__}")
import torchvision
print(f"torchvision: {torchvision.__version__}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device — confirm SM 6.0-compatible torch is active
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # surface kernel errors at the exact op

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    cuda_ver = torch.version.cuda
    torch_ver = torch.__version__
    print(f'[OK] Device      : {DEVICE}')
    print(f'   GPU           : {gpu_name}')
    print(f'   CUDA version  : {cuda_ver}')
    print(f'   torch version : {torch_ver}')
    # Verify torch was built with SM 6.0 support (P100)
    try:
        _t = torch.zeros(1).cuda()
        del _t
        print('[OK] CUDA kernel smoke test passed — SM 6.0 support confirmed')
    except Exception as e:
        raise RuntimeError(
            f'CUDA kernel test failed: {e}\n'
            'torch was not built with SM 6.0 support. '
            'Make sure Cell 25 ran successfully and the kernel was restarted after install.'
        )
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
else:
    raise RuntimeError('No GPU found. Go to Kaggle Settings -> Accelerator -> GPU P100.')


# CELL 2  -  Configuration

In [ ]:
# ── KAGGLE PATH CONFIG + CORRECTED EPOCHS (replaces Cell 2 in phase_2.ipynb) ──
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/phase1_outputs")
SPLITS_DIR = OUTPUT_DIR / "splits"
CGAN_DIR   = OUTPUT_DIR / "cgan"
SYNTH_DIR  = OUTPUT_DIR / "synthetic_images"

for d in [CGAN_DIR, SYNTH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE    = 64
LATENT_DIM    = 128
N_CLASSES_DX  = 7
EMBED_DIM     = 32
BATCH_SIZE    = 64
N_EPOCHS      = 200    # ← CRITICAL: was 5, must be 200+
LR_G          = 0.0002
LR_D          = 0.0002
BETA1         = 0.5
BETA2         = 0.999
GP_LAMBDA     = 10
N_CRITIC      = 5
SAVE_INTERVAL = 50

AGE_GROUPS = ["Elderly", "MiddleAged", "Pediatric", "YoungAdult"]
SEX_LABELS = ["female", "male"]
LOC_ZONES  = ["Head", "LowerExtremity", "Other", "Trunk", "Unknown", "UpperExtremity"]
DX_CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]

SYNTH_PER_SUBGROUP = 60

print(f"[OK] Config: {N_EPOCHS} epochs, batch={BATCH_SIZE}, image={IMAGE_SIZE}x{IMAGE_SIZE}")

# CELL 3  -  Load Data & Build Encoders

In [ ]:
df_train = pd.read_csv(SPLITS_DIR / "train_split.csv")
rare_subgroups = pd.read_csv(OUTPUT_DIR / "rare_subgroups_cgan_targets.csv")

print(f"Training split loaded: {len(df_train)} records")
print(f"Rare subgroups to augment: {len(rare_subgroups)}")

# - Label encoders (must match Step 1 feature engineering) -
dx_to_idx  = {v: i for i, v in enumerate(DX_CLASSES)}
age_to_idx = {v: i for i, v in enumerate(AGE_GROUPS)}
sex_to_idx = {v: i for i, v in enumerate(SEX_LABELS)}
loc_to_idx = {v: i for i, v in enumerate(LOC_ZONES)}

# Apply to training df
df_train["dx_idx"]  = df_train["dx"].map(dx_to_idx)
df_train["age_idx"] = df_train["age_group"].map(age_to_idx)
df_train["sex_idx"] = df_train["sex"].map(sex_to_idx)
df_train["loc_idx"] = df_train["loc_zone"].map(loc_to_idx)

# Drop any rows with unmapped values
df_train = df_train.dropna(subset=["dx_idx","age_idx","sex_idx","loc_idx"])
df_train = df_train.astype({"dx_idx":"int","age_idx":"int","sex_idx":"int","loc_idx":"int"})

print(f"\ndf_train after encoding: {len(df_train)} records")
print(df_train[["image_id","dx","age_group","sex","loc_zone",
                   "dx_idx","age_idx","sex_idx","loc_idx"]].head(5))

# CELL 4  -  Dataset Class

In [ ]:
class HAMDermDataset(Dataset):
    """
    Loads HAM10000 images with their condition labels for cGAN training.
    Returns: (image_tensor, conditions_dict)
    """
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (128, 128, 128))

        if self.transform:
            img = self.transform(img)

        conditions = {
            "dx"  : torch.tensor(row["dx_idx"],  dtype=torch.long),
            "age" : torch.tensor(row["age_idx"], dtype=torch.long),
            "sex" : torch.tensor(row["sex_idx"], dtype=torch.long),
            "loc" : torch.tensor(row["loc_idx"], dtype=torch.long),
        }
        return img, conditions


# - Transforms -
transform_train = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # -> [-1, 1]
])

dataset = HAMDermDataset(df_train, transform=transform_train)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=0, pin_memory=True if DEVICE.type=="cuda" else False,
                        drop_last=True)

print(f"[OK] Dataset: {len(dataset)} samples | {len(dataloader)} batches per epoch")

# CELL 5  -  cGAN Architecture: Condition Embedding Module

In [ ]:
class ConditionEmbedder(nn.Module):
    """
    Embeds all four condition labels (dx, age, sex, loc) into a single
    concatenated condition vector used by both Generator and Discriminator.
    """
    def __init__(self, n_dx=N_CLASSES_DX, n_age=len(AGE_GROUPS),
                 n_sex=len(SEX_LABELS), n_loc=len(LOC_ZONES),
                 embed_dim=EMBED_DIM):
        super().__init__()
        self.emb_dx  = nn.Embedding(n_dx,  embed_dim)
        self.emb_age = nn.Embedding(n_age, embed_dim)
        self.emb_sex = nn.Embedding(n_sex, embed_dim)
        self.emb_loc = nn.Embedding(n_loc, embed_dim)
        self.out_dim = embed_dim * 4  # 4 features x embed_dim each

    def forward(self, dx, age, sex, loc):
        return torch.cat([
            self.emb_dx(dx),
            self.emb_age(age),
            self.emb_sex(sex),
            self.emb_loc(loc),
        ], dim=1)


# CELL 6  -  cGAN Architecture: Generator

In [ ]:
class Generator(nn.Module):
    """
    DCGAN-style conditional generator.
    Input : noise vector z (LATENT_DIM) + condition embedding (EMBED_DIM*4)
    Output: RGB image (3 x IMAGE_SIZE x IMAGE_SIZE), values in [-1, 1]
    """
    def __init__(self):
        super().__init__()
        self.embedder = ConditionEmbedder()
        in_dim = LATENT_DIM + self.embedder.out_dim

        # Project + reshape: in_dim -> 512 x 4 x 4
        self.project = nn.Sequential(
            nn.Linear(in_dim, 512 * 4 * 4),
            nn.BatchNorm1d(512 * 4 * 4),
            nn.ReLU(True),
        )

        # Upsample blocks: 4x4 -> 8 -> 16 -> 32 -> 64
        self.conv_blocks = nn.Sequential(
            # 4x4 -> 8x8
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            # 8x8 -> 16x16
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # 16x16 -> 32x32
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # 32x32 -> 64x64
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh(),  # output in [-1, 1]
        )

    def forward(self, z, dx, age, sex, loc):
        cond   = self.embedder(dx, age, sex, loc)
        x      = torch.cat([z, cond], dim=1)
        x      = self.project(x)
        x      = x.view(x.size(0), 512, 4, 4)
        return self.conv_blocks(x)


# CELL 7  -  cGAN Architecture: Discriminator

In [ ]:
class Discriminator(nn.Module):
    """
    DCGAN-style conditional discriminator.
    Input : RGB image (3 x IMAGE_SIZE x IMAGE_SIZE) + projected condition map
    Output: scalar score (higher = more real)  -  unbounded for WGAN-GP
    """
    def __init__(self):
        super().__init__()
        self.embedder = ConditionEmbedder()
        # Project condition into a spatial map (1 x IMAGE_SIZE x IMAGE_SIZE)
        self.cond_proj = nn.Linear(self.embedder.out_dim, IMAGE_SIZE * IMAGE_SIZE)

        # Conv blocks operating on (3+1) = 4 input channels
        self.conv_blocks = nn.Sequential(
            # 64x64 -> 32x32
            nn.Conv2d(4, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # 32x32 -> 16x16
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # 16x16 -> 8x8
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # 8x8 -> 4x4
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            # 4x4 -> 1x1
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
        )

    def forward(self, img, dx, age, sex, loc):
        cond = self.embedder(dx, age, sex, loc)
        cond_map = self.cond_proj(cond).view(img.size(0), 1, IMAGE_SIZE, IMAGE_SIZE)
        x = torch.cat([img, cond_map], dim=1)  # (B, 4, 64, 64)
        return self.conv_blocks(x).view(img.size(0), -1).mean(1)  # scalar per sample

# CELL 8  -  WGAN-GP Gradient Penalty

In [ ]:
def compute_gradient_penalty(D, real_imgs, fake_imgs, dx, age, sex, loc, device):
    """
    WGAN-GP gradient penalty: enforces the 1-Lipschitz constraint on D.
    Penalises gradients that deviate from norm 1 along interpolated samples.
    """
    B = real_imgs.size(0)
    # Random interpolation coefficient
    alpha = torch.rand(B, 1, 1, 1, device=device)
    interp = (alpha * real_imgs + (1 - alpha) * fake_imgs).requires_grad_(True)

    d_interp = D(interp, dx, age, sex, loc)
    gradients = torch.autograd.grad(
        outputs=d_interp,
        inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    gradients = gradients.view(B, -1)
    gp = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gp

# CELL 9  -  Initialise Models, Optimisers

In [ ]:
def weights_init(m):
    """DCGAN weight initialisation: Conv -> N(0, 0.02), BatchNorm -> N(1, 0.02)"""
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)

G.apply(weights_init)
D.apply(weights_init)

opt_G = optim.Adam(G.parameters(), lr=LR_G, betas=(BETA1, BETA2))
opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=(BETA1, BETA2))

# Count parameters
n_params_G = sum(p.numel() for p in G.parameters() if p.requires_grad)
n_params_D = sum(p.numel() for p in D.parameters() if p.requires_grad)
print(f"[OK] Models initialised.")
print(f"   Generator     : {n_params_G:,} trainable parameters")
print(f"   Discriminator : {n_params_D:,} trainable parameters")

# Fixed noise for visual monitoring during training
fixed_noise = torch.randn(16, LATENT_DIM, device=DEVICE)
# Use a varied set of conditions for the fixed samples
fixed_conds = {
    "dx"  : torch.tensor([0,1,2,3,4,5,6,0,1,2,3,4,5,6,0,1], device=DEVICE),
    "age" : torch.tensor([0,1,2,3,0,1,2,3,0,1,2,3,0,1,2,3], device=DEVICE),
    "sex" : torch.tensor([0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1], device=DEVICE),
    "loc" : torch.tensor([0,1,2,3,4,5,0,1,2,3,4,5,0,1,2,3], device=DEVICE),
}


# CELL 10  -  Training Loop (WGAN-GP)

In [ ]:
# - Training history -
history = {"epoch":[], "d_loss":[], "g_loss":[], "gp":[]}

print("=" * 60)
print(f"  TRAINING cGAN ({N_EPOCHS} epochs, WGAN-GP)")
print("=" * 60)

for epoch in range(1, N_EPOCHS + 1):
    G.train(); D.train()
    epoch_d_loss, epoch_g_loss, epoch_gp = 0.0, 0.0, 0.0
    n_batches = 0

    for real_imgs, conds in dataloader:
        real_imgs = real_imgs.to(DEVICE)
        dx  = conds["dx"].to(DEVICE)
        age = conds["age"].to(DEVICE)
        sex = conds["sex"].to(DEVICE)
        loc = conds["loc"].to(DEVICE)
        B   = real_imgs.size(0)

        # - Train Discriminator N_CRITIC times per generator step -
        for _ in range(N_CRITIC):
            z = torch.randn(B, LATENT_DIM, device=DEVICE)
            fake_imgs = G(z, dx, age, sex, loc).detach()

            d_real = D(real_imgs, dx, age, sex, loc)
            d_fake = D(fake_imgs, dx, age, sex, loc)
            gp     = compute_gradient_penalty(D, real_imgs, fake_imgs,
                                              dx, age, sex, loc, DEVICE)
            d_loss = d_fake.mean() - d_real.mean() + GP_LAMBDA * gp

            opt_D.zero_grad()
            d_loss.backward()
            opt_D.step()

        # - Train Generator -
        z = torch.randn(B, LATENT_DIM, device=DEVICE)
        fake_imgs = G(z, dx, age, sex, loc)
        g_loss    = -D(fake_imgs, dx, age, sex, loc).mean()

        opt_G.zero_grad()
        g_loss.backward()
        opt_G.step()

        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()
        epoch_gp     += gp.item()
        n_batches    += 1

    avg_d = epoch_d_loss / n_batches
    avg_g = epoch_g_loss / n_batches
    avg_gp = epoch_gp / n_batches
    history["epoch"].append(epoch)
    history["d_loss"].append(avg_d)
    history["g_loss"].append(avg_g)
    history["gp"].append(avg_gp)

    # - Logging -
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch [{epoch:>4}/{N_EPOCHS}] | "
              f"D Loss: {avg_d:>8.4f} | G Loss: {avg_g:>8.4f} | GP: {avg_gp:.4f}")

    # - Visualise generated samples -
    if epoch % SAVE_INTERVAL == 0 or epoch == N_EPOCHS:
        G.eval()
        with torch.no_grad():
            sample_imgs = G(fixed_noise,
                            fixed_conds["dx"], fixed_conds["age"],
                            fixed_conds["sex"], fixed_conds["loc"])
        grid = vutils.make_grid(sample_imgs, nrow=4, normalize=True, value_range=(-1,1))
        grid_np = grid.permute(1, 2, 0).cpu().numpy()
        plt.figure(figsize=(10, 6))
        plt.imshow(grid_np)
        plt.axis("off")
        plt.title(f"cGAN Generated Samples  -  Epoch {epoch}", fontweight="bold")
        plt.tight_layout()
        plt.savefig(CGAN_DIR / f"samples_epoch_{epoch:04d}.png", dpi=120, bbox_inches="tight")
        plt.show()

        # Save checkpoint
        torch.save({"epoch": epoch,
                    "G_state": G.state_dict(),
                    "D_state": D.state_dict(),
                    "opt_G": opt_G.state_dict(),
                    "opt_D": opt_D.state_dict()},
                   CGAN_DIR / f"checkpoint_epoch_{epoch:04d}.pt")
        print(f"   [OK] Checkpoint saved -> epoch {epoch}")

# Save final model
torch.save(G.state_dict(), CGAN_DIR / "generator_final.pt")
torch.save(D.state_dict(), CGAN_DIR / "discriminator_final.pt")
print("\n[OK] cGAN training complete. Final models saved.")

# CELL 11  -  Plot Training Losses

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.plot(history["epoch"], history["d_loss"], label="Discriminator Loss", color="#e74c3c")
ax1.plot(history["epoch"], history["g_loss"], label="Generator Loss",     color="#3498db")
ax1.set_title("cGAN Training Losses (WGAN-GP)", fontweight="bold")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend()

ax2.plot(history["epoch"], history["gp"], label="Gradient Penalty", color="#27ae60")
ax2.set_title("Gradient Penalty Over Training", fontweight="bold")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("GP Value"); ax2.legend()

plt.tight_layout()
plt.savefig(CGAN_DIR / "training_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("[OK] Loss curves saved.")

# CELL 12  -  FID Score Helper (Quality Filter)

In [ ]:
def compute_fid_score(real_features, fake_features):
    """
    Frechet Inception Distance between real and generated image distributions.
    Lower FID = better quality (closer to real distribution).
    """
    mu1, sigma1 = real_features.mean(0), np.cov(real_features, rowvar=False)
    mu2, sigma2 = fake_features.mean(0), np.cov(fake_features, rowvar=False)

    diff = mu1 - mu2
    # Matrix square root of sigma1 @ sigma2
    covmean, _ = linalg.sqrtm(sigma1 @ sigma2, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return float(fid)


def extract_inception_features(imgs_tensor, inception_model, device, batch_size=32):
    """
    Pass images through Inception-v3 (pool3 layer) to get 2048-D feature vectors.
    imgs_tensor: (N, 3, H, W) in [-1,1]
    """
    inception_model.eval()
    # Inception expects 299x299
    resize_fn = transforms.Resize((299, 299))
    features = []

    for i in range(0, len(imgs_tensor), batch_size):
        batch = imgs_tensor[i:i+batch_size]
        batch = torch.stack([resize_fn(img) for img in batch]).to(device)
        # Normalise from [-1,1] -> [0,1] -> Inception normalisation
        batch = (batch + 1) / 2
        batch = transforms.Normalize(mean=[0.485,0.456,0.406],
                                     std=[0.229,0.224,0.225])(batch)
        with torch.no_grad():
            feat = inception_model(batch)
        features.append(feat.cpu().numpy())

    return np.concatenate(features, axis=0)


# Load a light Inception model for FID (pool_aux disabled)
print("Loading Inception-v3 for FID computation...")
inception = inception_v3(pretrained=True, transform_input=False)
inception.fc = nn.Identity()  # Use 2048-D pool features instead of 1000-class output
inception.aux_logits = False
inception = inception.to(DEVICE)
print("[OK] Inception-v3 loaded.")

# CELL 13  -  Generate Synthetic Images for Rare Subgroups 

In [ ]:
print("=" * 60)
print("  GENERATING SYNTHETIC IMAGES FOR RARE SUBGROUPS")
print("=" * 60)

G.eval()
G.load_state_dict(torch.load(CGAN_DIR / "generator_final.pt", map_location=DEVICE))

# - Extract real image features for FID baseline -
print("\nExtracting real image features for FID baseline...")
real_tensors = []
transform_eval = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
sample_ids = df_train.sample(min(500, len(df_train)), random_state=SEED)["image_path"].tolist()
for p in tqdm(sample_ids, desc="Loading real images"):
    try:
        img = Image.open(p).convert("RGB")
        real_tensors.append(transform_eval(img))
    except:
        pass
real_tensors = torch.stack(real_tensors)
real_features = extract_inception_features(real_tensors, inception, DEVICE)

# - Generate images per rare subgroup -
generation_log = []

for _, row in tqdm(rare_subgroups.iterrows(), total=len(rare_subgroups),
                   desc="Processing rare subgroups"):
    dx_code   = row["dx"]
    age_grp   = row["age_group"]
    sex_val   = row["sex"]
    loc_zone  = row["loc_zone"]
    key       = row["intersectional_key"]
    existing  = int(row["count"])

    # Skip if any label is unknown/unmapped
    if any([dx_code not in dx_to_idx, age_grp not in age_to_idx,
            sex_val not in sex_to_idx, loc_zone not in loc_to_idx]):
        print(f"[WARN]  Skipping unmapped subgroup: {key}")
        continue

    # Determine how many images to generate
    n_to_generate = max(SYNTH_PER_SUBGROUP, 60 - existing)

    # Build condition tensors
    dx_t  = torch.tensor([dx_to_idx[dx_code]],  dtype=torch.long, device=DEVICE).repeat(n_to_generate)
    age_t = torch.tensor([age_to_idx[age_grp]], dtype=torch.long, device=DEVICE).repeat(n_to_generate)
    sex_t = torch.tensor([sex_to_idx[sex_val]], dtype=torch.long, device=DEVICE).repeat(n_to_generate)
    loc_t = torch.tensor([loc_to_idx[loc_zone]],dtype=torch.long, device=DEVICE).repeat(n_to_generate)

    # Generate in batches
    generated = []
    with torch.no_grad():
        for start in range(0, n_to_generate, 32):
            end = min(start + 32, n_to_generate)
            z = torch.randn(end - start, LATENT_DIM, device=DEVICE)
            imgs = G(z, dx_t[start:end], age_t[start:end],
                        sex_t[start:end], loc_t[start:end])
            generated.append(imgs.cpu())
    generated = torch.cat(generated, dim=0)

    # - FID Quality Filtering -
    fake_features = extract_inception_features(generated, inception, DEVICE)
    fid = compute_fid_score(real_features, fake_features)

    # Accept all if FID is reasonable; otherwise keep top half by discriminator score
    D.eval()
    with torch.no_grad():
        d_scores = D(generated.to(DEVICE), dx_t[:len(generated)],
                     age_t[:len(generated)], sex_t[:len(generated)],
                     loc_t[:len(generated)]).cpu().numpy()

    # Sort by discriminator score (descending = more realistic)
    sorted_indices = np.argsort(d_scores)[::-1].copy()
    # Keep top 50% or at most 40 images
    n_keep = min(40, max(20, len(sorted_indices) // 2))
    keep_indices = sorted_indices[:n_keep]
    accepted_imgs = generated[keep_indices]

    # - Save accepted synthetic images -
    subgroup_dir = SYNTH_DIR / key.replace("|", "_")
    subgroup_dir.mkdir(parents=True, exist_ok=True)

    for i, img_tensor in enumerate(accepted_imgs):
        # Denormalize from [-1,1] -> [0,255]
        img_np = ((img_tensor.permute(1,2,0).numpy() + 1) / 2 * 255).astype(np.uint8)
        img_pil = Image.fromarray(img_np)
        # Upsample to 224x224 for CNN training compatibility
        img_pil_224 = img_pil.resize((224, 224), Image.BILINEAR)
        save_path = subgroup_dir / f"synth_{key.replace('|','_')}_{i:04d}.jpg"
        img_pil_224.save(save_path, quality=95)

    generation_log.append({
        "intersectional_key": key, "dx": dx_code, "age_group": age_grp,
        "sex": sex_val, "loc_zone": loc_zone,
        "existing_count": existing, "generated": n_to_generate,
        "accepted_after_filter": n_keep, "fid_score": round(fid, 2)
    })
    print(f"  [OK] {key:<40} | generated:{n_to_generate:>3} | "
          f"kept:{n_keep:>3} | FID:{fid:.1f}")

# - Save generation log -
gen_log_df = pd.DataFrame(generation_log)
gen_log_df.to_csv(OUTPUT_DIR / "cgan_generation_log.csv", index=False)
print(f"\n[OK] Generation complete. Log saved -> {OUTPUT_DIR / 'cgan_generation_log.csv'}")

# CELL 14  -  Build Augmented Training Manifest

In [ ]:
# - Collect original training images -
original_manifest = df_train[["image_id","image_path","dx","age_group",
                               "sex","loc_zone","intersectional_key",
                               "sample_weight"]].copy()
original_manifest["is_synthetic"] = False

# - Collect synthetic images -
synth_rows = []
for _, row in gen_log_df.iterrows():
    key  = row["intersectional_key"]
    subgroup_dir = SYNTH_DIR / key.replace("|", "_")
    for img_file in subgroup_dir.glob("*.jpg"):
        synth_rows.append({
            "image_id"          : img_file.stem,
            "image_path"        : str(img_file),
            "dx"                : row["dx"],
            "age_group"         : row["age_group"],
            "sex"               : row["sex"],
            "loc_zone"          : row["loc_zone"],
            "intersectional_key": key,
            "sample_weight"     : 1.0,  # synthetic images get baseline weight
            "is_synthetic"      : True,
        })

synth_manifest = pd.DataFrame(synth_rows)

# - Combine into augmented training set (v2 dataset) -
df_train_augmented = pd.concat([original_manifest, synth_manifest], ignore_index=True)
df_train_augmented.to_csv(SPLITS_DIR / "train_split_augmented_v2.csv", index=False)

print(f"\n[CHART] AUGMENTED TRAINING SET (v2):")
print(f"  Original training images : {len(original_manifest):,}")
print(f"  Synthetic images added   : {len(synth_manifest):,}")
print(f"  Total augmented dataset  : {len(df_train_augmented):,}")
print(f"\n[OK] Augmented manifest saved -> {SPLITS_DIR / 'train_split_augmented_v2.csv'}")

print(f"\n  NEXT STEP -> Open: phase1_step3_cnn_training_evaluation.py")

# End of Part 1

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 1+2 FINAL CELL — Package all outputs for Kaggle Dataset
# Run this after cGAN training completes.
# ═══════════════════════════════════════════════════════════════
import json, shutil, os
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/phase1_outputs")

print("=" * 55)
print("  PACKAGING PHASE 1+2 OUTPUTS")
print("=" * 55)

# ── Verify critical files exist ───────────────────────────────
checks = {
    "train_split.csv"              : OUTPUT_DIR / "splits" / "train_split.csv",
    "val_split.csv"                : OUTPUT_DIR / "splits" / "val_split.csv",
    "test_split.csv"               : OUTPUT_DIR / "splits" / "test_split.csv",
    "train_split_augmented_v2.csv" : OUTPUT_DIR / "splits" / "train_split_augmented_v2.csv",
    "label_map.json"               : OUTPUT_DIR / "label_map.json",
    "rare_subgroups.csv"           : OUTPUT_DIR / "rare_subgroups_cgan_targets.csv",
    "cgan_generation_log.csv"      : OUTPUT_DIR / "cgan_generation_log.csv",
    "sample_weights.json"          : OUTPUT_DIR / "sample_weights.json",
}

all_ok = True
for name, path in checks.items():
    if path.exists():
        size = path.stat().st_size / 1024
        print(f"  [OK] {name:<40} {size:>8.1f} KB")
    else:
        print(f"  [MISSING] {name}")
        all_ok = False

# Count synthetic images
synth_dir = OUTPUT_DIR / "synthetic_images"
n_synth = sum(1 for _ in synth_dir.rglob("*.jpg")) if synth_dir.exists() else 0
print(f"\n  Synthetic images generated: {n_synth:,}")

# ── Save a path manifest so Phase 3 notebook can remap paths ──
# This records where synthetic images are stored inside this output,
# so Notebook 2 can find them after the dataset is mounted.
manifest = {
    "phase12_base"          : str(OUTPUT_DIR),
    "splits_dir"            : str(OUTPUT_DIR / "splits"),
    "synthetic_images_dir"  : str(synth_dir),
    "label_map"             : str(OUTPUT_DIR / "label_map.json"),
    "sample_weights"        : str(OUTPUT_DIR / "sample_weights.json"),
    "rare_subgroups"        : str(OUTPUT_DIR / "rare_subgroups_cgan_targets.csv"),
    "cgan_log"              : str(OUTPUT_DIR / "cgan_generation_log.csv"),
    "n_synthetic_images"    : n_synth,
}
with open(OUTPUT_DIR / "phase12_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print(f"  [OK] phase12_manifest.json saved")